# UC-11 — seed_bronze (Databricks)

Genereert deterministische synthetic data voor zeven UC-11 bron-domeinen en schrijft
ze als UC-managed Delta-tabellen onder `databricks.bronze.*`.

**Idempotent**: gebruikt `mode('overwrite')` met seed `42`. Dezelfde input → identieke output
(matchend met de Fabric-variant zodat we cross-engine kunnen vergelijken).

SYNTHETIC DATA — OPEN DATA AND ANALYTICS PLATFORM — NOT FOR REAL USE.

In [ ]:
dbutils.widgets.text("catalog", "databricks")
dbutils.widgets.text("seed", "42")
dbutils.widgets.text("n_personen", "200")

catalog = dbutils.widgets.get("catalog")
seed = int(dbutils.widgets.get("seed"))
n_personen = int(dbutils.widgets.get("n_personen"))

spark.sql(f"USE CATALOG {catalog}")
spark.sql("USE SCHEMA bronze")
print(f"Schrijven naar {catalog}.bronze, n_personen={n_personen}")

In [ ]:
import random
from datetime import datetime, timedelta
from pyspark.sql import Row

rnd = random.Random(seed)

VOORNAMEN = ["Jan", "Piet", "Marie", "Anna", "Mohammed", "Fatima", "Sven", "Lieke",
             "Ahmet", "Esra", "Ravi", "Priya", "Lars", "Sofie", "Bram", "Emma"]
ACHTERNAMEN = ["de Vries", "Jansen", "van Dijk", "Bakker", "Yilmaz", "El Amrani",
               "Visser", "Smit", "Mulder", "Kumar", "Singh", "de Boer"]
WOONPLAATSEN = ["Amsterdam", "Rotterdam", "Utrecht", "Den Haag", "Groningen",
                "Eindhoven", "Tilburg", "Arnhem", "Maastricht", "Leeuwarden"]
REGIO_CODES = ["NH", "ZH", "UT", "GR", "NB", "GE", "LI", "FR", "OV", "DR"]
WERKGEVERS = ["Albert Heijn", "PostNL", "Philips", "ASML", "NS", "Rabobank",
              "KPN", "Shell", "Ahold", "Heineken", "DSM", "Schiphol"]

T0 = datetime(2020, 1, 1)
def bsn(i): return f"1{i:08d}"
def rand_date(start_offset_days, span_days):
    return T0 + timedelta(days=start_offset_days + rnd.randint(0, span_days))

personen_ids = list(range(1, n_personen + 1))

In [ ]:
# ─── persoon ────────────────────────────────────────────────────────
persoon_rows = []
for i in personen_ids:
    geb = rand_date(-365 * 50, 365 * 30).date()
    persoon_rows.append(Row(
        bsn=bsn(i), voornaam=rnd.choice(VOORNAMEN),
        achternaam=rnd.choice(ACHTERNAMEN),
        geslacht=rnd.choice(["M", "V"]), geboortedatum=geb,
        straat=f"{rnd.choice(['Hoofdstraat','Kerkstraat','Schoolstraat'])} ",
        huisnummer=rnd.randint(1, 200),
        postcode=f"{rnd.randint(1000,9999)}AB",
        woonplaats=rnd.choice(WOONPLAATSEN),
        nationaliteit=rnd.choice(["NL", "NL", "NL", "MA", "TR", "DE", "PL"]),
        source_ts=rand_date(0, 1800),
        event_date=rand_date(0, 1800).date(),
    ))
df = spark.createDataFrame(persoon_rows)
df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.persona_created")
print(f"bronze.persona_created: {df.count()} rijen")

In [ ]:
# ─── polisadm.ikv ────────────────────────────────────────────────────
ikv_rows = []
ikv_id = 0
for i in personen_ids:
    for _ in range(rnd.randint(0, 3)):
        ikv_id += 1
        start = rand_date(0, 1500)
        duur = rnd.randint(60, 1000)
        eind = start + timedelta(days=duur) if rnd.random() < 0.6 else None
        ikv_rows.append(Row(
            ikv_id=f"IKV{ikv_id:08d}", bsn=bsn(i),
            werkgever_naam=rnd.choice(WERKGEVERS),
            aanvang_dienstverband=start.date(),
            einde_dienstverband=eind.date() if eind else None,
            loon_bruto_jaar=float(rnd.randint(25000, 95000)),
            source_ts=start, event_date=start.date(),
        ))
df = spark.createDataFrame(ikv_rows)
df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.polisadm_ikv")
print(f"bronze.polisadm_ikv: {df.count()} rijen")

In [ ]:
# ─── ww.aanvraag ────────────────────────────────────────────────────
ww_rows = []
for j, i in enumerate(rnd.sample(personen_ids, min(len(personen_ids) // 2, n_personen))):
    ts = rand_date(365, 1200)
    ww_rows.append(Row(
        aanvraag_id=f"WW{j+1:08d}", bsn=bsn(i),
        aanvraag_datum=ts.date(),
        status=rnd.choice(["toegekend", "toegekend", "afgewezen", "in behandeling"]),
        reden_einde_dienstverband=rnd.choice(["ontslag", "einde tijdelijk contract",
                                              "reorganisatie", "faillissement"]),
        source_ts=ts, event_date=ts.date(),
    ))
df = spark.createDataFrame(ww_rows)
df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.ww_aanvraag")
print(f"bronze.ww_aanvraag: {df.count()} rijen")

In [ ]:
# ─── zw.melding ─────────────────────────────────────────────────────
zw_rows = []
for j, i in enumerate(rnd.sample(personen_ids, n_personen * 2 // 5)):
    ts = rand_date(365, 1200)
    zw_rows.append(Row(
        melding_id=f"ZW{j+1:08d}", bsn=bsn(i),
        eerste_ziektedag=ts.date(),
        duur_dagen=rnd.randint(1, 730),
        source_ts=ts, event_date=ts.date(),
    ))
df = spark.createDataFrame(zw_rows)
df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.zw_melding")
print(f"bronze.zw_melding: {df.count()} rijen")

In [ ]:
# ─── wia.aanvraag ───────────────────────────────────────────────────
wia_rows = []
for j, i in enumerate(rnd.sample(personen_ids, n_personen // 4)):
    ts = rand_date(700, 800)
    wia_rows.append(Row(
        aanvraag_id=f"WIA{j+1:08d}", bsn=bsn(i),
        aanvraag_datum=ts.date(),
        status=rnd.choice(["toegekend", "afgewezen", "in behandeling"]),
        onderdeel=rnd.choice(["IVA", "WGA"]),
        arbeidsongeschikt_pct=float(rnd.choice([35, 45, 55, 65, 80, 100])),
        regio_code=rnd.choice(REGIO_CODES),
        source_ts=ts, event_date=ts.date(),
    ))
df = spark.createDataFrame(wia_rows)
df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.wia_aanvraag")
print(f"bronze.wia_aanvraag: {df.count()} rijen")

In [ ]:
# ─── wajong.dossier ─────────────────────────────────────────────────
wajong_rows = []
for j, i in enumerate(rnd.sample(personen_ids, n_personen // 7)):
    ts = rand_date(0, 1500)
    wajong_rows.append(Row(
        dossier_id=f"WJ{j+1:08d}", bsn=bsn(i),
        ingangsdatum=ts.date(),
        regime=rnd.choice(["oWajong", "Wajong2010", "Wajong2015"]),
        arbeidsvermogen=rnd.choice(["ja", "nee", "deels"]),
        source_ts=ts, event_date=ts.date(),
    ))
df = spark.createDataFrame(wajong_rows)
df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.wajong_dossier")
print(f"bronze.wajong_dossier: {df.count()} rijen")

In [ ]:
# ─── crm.contact ────────────────────────────────────────────────────
crm_rows = []
for j in range(n_personen * 3):
    i = rnd.choice(personen_ids)
    ts = rand_date(0, 1800)
    crm_rows.append(Row(
        contact_id=f"CRM{j+1:08d}", bsn=bsn(i), contact_ts=ts,
        kanaal=rnd.choice(["telefoon", "email", "chat", "balie", "telefoon"]),
        onderwerp=rnd.choice(["WW aanvraag", "WIA status", "betaling",
                              "reintegratie", "document opvragen", "klacht"]),
        duur_seconden=float(rnd.randint(30, 1800)),
        event_date=ts.date(),
    ))
df = spark.createDataFrame(crm_rows)
df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.crm_contact")
print(f"bronze.crm_contact: {df.count()} rijen")
print("Seed bronze klaar.")